<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class TeaEstateAnalyzer:
    def __init__(self, soil_df: pd.DataFrame, weather_df: pd.DataFrame):
        self.soil_df = soil_df.copy()
        self.weather_df = weather_df.copy()

        # Tea soil standards
        self.PH_LOWER_LIMIT = 4.5
        self.PH_UPPER_LIMIT = 5.5
        self.WILTING_POINT = 5
        self.SATURATION_POINT = 100

    # -------------------------------------------------
    # EVAPOTRANSPIRATION (Refined Approximation)
    # -------------------------------------------------
    def calculate_evapotranspiration(self, temp, humidity=70):
        """
        PET approximation:
        - Temperature increases evaporation exponentially
        - Humidity suppresses evaporation
        """
        temp_factor = 0.1 * (temp ** 1.05)
        humidity_factor = (1 - humidity / 100)
        return temp_factor * humidity_factor

    # -------------------------------------------------
    # MOISTURE PROJECTION
    # -------------------------------------------------
    def predict_moisture_trend(self, field_no):
        field_data = self.soil_df[self.soil_df['Field_No'] == field_no]

        if field_data.empty:
            raise ValueError(f"Field {field_no} not found")

        field = field_data.iloc[0]
        current_moisture = field['Moisture_Base']

        # Organic carbon improves water holding capacity
        absorption_coeff = 0.30 + (field['Carbon_Pct'] * 0.05)

        projections = []

        for _, day in self.weather_df.iterrows():
            evap = self.calculate_evapotranspiration(
                temp=day['temp'],
                humidity=day.get('humidity', 70)
            )

            # Rainfall effectiveness
            effective_rain = min(day['rain'], 35) + max(0, day['rain'] - 35) * 0.1

            # Moisture balance
            new_moisture = (
                current_moisture
                - evap
                + (effective_rain * absorption_coeff)
            )

            current_moisture = np.clip(
                new_moisture,
                self.WILTING_POINT,
                self.SATURATION_POINT
            )

            projections.append({
                'Date': day['date'],
                'Rain_mm': day['rain'],
                'Temp_C': day['temp'],
                'Evap_mm': round(evap, 2),
                'Soil_Moisture_%': round(current_moisture, 2)
            })

        return pd.DataFrame(projections)

    # -------------------------------------------------
    # SOIL HEALTH SCORE (0–100)
    # -------------------------------------------------
    def calculate_soil_health_score(self, field):
        ph_score = max(0, 100 - abs(5.0 - field['pH']) * 40)
        carbon_score = min(field['Carbon_Pct'] / 2.5 * 100, 100)
        moisture_score = min(field['Moisture_Base'] / 35 * 100, 100)

        return round((ph_score + carbon_score + moisture_score) / 3, 1)

    # -------------------------------------------------
    # DECISION ENGINE
    # -------------------------------------------------
    def generate_management_report(self, field_no):
        field = self.soil_df[self.soil_df['Field_No'] == field_no].iloc[0]
        projections = self.predict_moisture_trend(field_no)

        avg_moisture = projections['Soil_Moisture_%'].mean()
        total_rain = projections['Rain_mm'].sum()
        soil_health = self.calculate_soil_health_score(field)

        # Decision rules
        if field['pH'] < self.PH_LOWER_LIMIT:
            decision = "STOP – Apply Dolomite (pH too acidic)"
        elif total_rain > 60:
            decision = "DELAY – Heavy rain forecast (leaching risk)"
        elif avg_moisture < 18:
            decision = "IRRIGATION REQUIRED"
        elif field['Carbon_Pct'] < 1.8:
            decision = "Apply Fertilizer + Compost/Biochar"
        else:
            decision = "Proceed with planned NPK application"

        return {
            "Field_No": field_no,
            "Soil_Health_Score": soil_health,
            "Avg_Moisture_%": round(avg_moisture, 2),
            "Total_Rain_mm": total_rain,
            "Decision": decision,
            "Projection_Table": projections
        }

# -------------------------------------------------
# DATA INITIALIZATION
# -------------------------------------------------
soil_data = {
    'Field_No': [1, 2, 3, 4],
    'pH': [4.4, 4.6, 5.1, 3.8],
    'Carbon_Pct': [1.8, 1.5, 2.1, 1.1],
    'Moisture_Base': [28, 25, 30, 22]
}

forecast = {
    'date': [(datetime.now() + timedelta(days=i)).date() for i in range(5)],
    'rain': [12, 0, 45, 5, 2],
    'temp': [24, 25, 22, 26, 28]
}

# -------------------------------------------------
# EXECUTION
# -------------------------------------------------
analyzer = TeaEstateAnalyzer(
    pd.DataFrame(soil_data),
    pd.DataFrame(forecast)
)

report = analyzer.generate_management_report(field_no=4)

print(f"\n📋 FIELD {report['Field_No']} MANAGEMENT REPORT")
print(f"Soil Health Score: {report['Soil_Health_Score']}/100")
print(f"Decision: {report['Decision']}\n")
print(report['Projection_Table'].to_string(index=False))



📋 FIELD 4 MANAGEMENT REPORT
Soil Health Score: 53.0/100
Decision: STOP – Apply Dolomite (pH too acidic)

      Date  Rain_mm  Temp_C  Evap_mm  Soil_Moisture_%
2025-12-30       12      24     0.84            25.42
2025-12-31        0      25     0.88            24.54
2026-01-01       45      22     0.77            36.54
2026-01-02        5      26     0.92            37.40
2026-01-03        2      28     0.99            37.12
